# Decision-payoff selection vs. statistical accuracy

A simple binary-gamble example to make one point: **the model that best fits the data statistically is not always the model that gives the best decision.** The agent's risk aversion $\gamma$ controls which trade-off they care about, so different agents can rationally select different models even when they share the same data.

**Setup.** The true gamble pays multiple $M_+ = 1.5$ with probability $p = 0.55$ and multiple $M_- = 0.7$ otherwise. Each period the agent invests fraction $f$ of wealth, so per-period wealth is
$$
W_{t+1} = W_t \cdot \bigl[1 + f(M - 1)\bigr], \qquad M \in \{M_+, M_-\}.
$$

**Three candidate models** of the gamble, each disagreeing with the truth on either $q$, the multiples, or both:

| Model | $q$ | $M_+$ | $M_-$ | character |
|---|---:|---:|---:|---|
| A — lottery | 0.30 | 3.0 | 0.7 | thinks wins are rare but big |
| B — truth   | 0.55 | 1.5 | 0.7 | exactly the truth |
| C — insurance | 0.80 | 1.2 | 0.7 | thinks wins are common but small |

**Three agents** with CRRA risk aversion $\gamma \in \{0.5, 1, 2\}$. Each agent considers all three models, picks an $f^\star$ under each, and selects among the models by **realized mean utility on a validation set** of 1000 draws from the true gamble. We then ask whether the winning model differs across agents.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

rng = np.random.default_rng(2026)

truth = {"p": 0.55, "M_plus": 1.5, "M_minus": 0.7}
models = {
    "A (lottery)":   {"q": 0.30, "M_plus": 3.0, "M_minus": 0.7},
    "B (truth)":     {"q": 0.55, "M_plus": 1.5, "M_minus": 0.7},
    "C (insurance)": {"q": 0.80, "M_plus": 1.2, "M_minus": 0.7},
}
gammas = [0.5, 1.0, 2.0]

## Training stage: each agent computes $f^\star$ under each model

Each agent solves
$$
f^\star(\gamma, \text{model}) = \arg\max_f \; \mathbb{E}_{\text{model}}\bigl[u_\gamma(1 + f(M - 1))\bigr]
$$
where the expectation is taken under the model's beliefs $(q, M_+, M_-)$. The search runs up to the model-implied ruin boundary $f < 1/(1 - M_-)$ so that wealth stays strictly positive in the loss state.

In [2]:
def neg_eu(f, q, Mp, Mm, gamma):
    wW = 1.0 + f * (Mp - 1.0)
    wL = 1.0 + f * (Mm - 1.0)
    if wW <= 0 or wL <= 0:
        return 1e10
    if gamma == 1:
        return -(q * np.log(wW) + (1 - q) * np.log(wL))
    return -(q * (wW ** (1 - gamma) - 1) / (1 - gamma)
             + (1 - q) * (wL ** (1 - gamma) - 1) / (1 - gamma))

def f_star(model, gamma, f_max=5.0):
    q, Mp, Mm = model["q"], model["M_plus"], model["M_minus"]
    f_ruin = 1.0 / (1 - Mm) if Mm < 1 else f_max
    upper = min(f_ruin - 1e-4, f_max)
    res = minimize_scalar(neg_eu, args=(q, Mp, Mm, gamma),
                          bounds=(0.0, upper), method="bounded",
                          options={"xatol": 1e-7})
    return res.x

fstar_table = pd.DataFrame(
    {name: [f_star(m, g) for g in gammas] for name, m in models.items()},
    index=pd.Index(gammas, name="gamma"),
).round(3)
fstar_table

,A (lottery),B (truth),C (insurance)
gamma,,,
0.5,1.610,1.805,2.619
1.0,0.650,0.933,1.667
2.0,0.275,0.460,0.918


## Validation stage: 1000 draws from the truth

Simulate 1000 i.i.d. realizations of the true gamble. For each $(\gamma, \text{model})$ pair, compute realized mean utility
$$
\frac{1}{n} \sum_{t} u_\gamma\bigl(1 + f^\star_{\gamma, \text{model}}\,(M_t - 1)\bigr)
$$
where $M_t$ is the realized payoff under the truth (not under the model). The winning model per $\gamma$ is the one whose recommended $f^\star$ delivers the highest realized utility on data drawn from the true process — *not* the one whose beliefs are closest to the truth.

In [3]:
n_draws = 1000
wins = rng.random(n_draws) < truth["p"]
payoffs = np.where(wins, truth["M_plus"], truth["M_minus"])
print(f"realized win frequency: {wins.mean():.3f}  (truth p = {truth['p']})")

def realized_mean_utility(f, payoffs, gamma):
    w = 1.0 + f * (payoffs - 1.0)
    if np.any(w <= 0):
        return -np.inf
    if gamma == 1:
        return np.log(w).mean()
    return ((w ** (1 - gamma) - 1) / (1 - gamma)).mean()

val_table = pd.DataFrame(
    {name: [realized_mean_utility(fstar_table.loc[g, name], payoffs, g) for g in gammas]
     for name in models},
    index=pd.Index(gammas, name="gamma"),
)
winners = val_table.idxmax(axis=1).rename("winner")
pd.concat([val_table.round(5), winners], axis=1)

realized win frequency: 0.516  (truth p = 0.55)


,A (lottery),B (truth),C (insurance),winner
gamma,,,,
0.5,0.08251,0.07891,0.01645,A (lottery)
1.0,0.04022,0.03864,-0.02277,A (lottery)
2.0,0.01885,0.01900,-0.02162,B (truth)


## What we just saw

**Different agents picked different models.** The lottery model A — whose stated win probability $q = 0.30$ is far from the truth's $p = 0.55$ — wins at low and moderate $\gamma$. The truth-matching model B wins only at high $\gamma$.

**Why this happens.** Each model's $f^\star$ depends on both its claimed mean (the product $q \cdot M_+$) *and* its claimed loss state $M_-$. Models that disagree about the magnitude of the upside but agree about the loss state recommend very different $f^\star$ values, and those values have different sensitivities to the *realized* loss frequency. At low $\gamma$ the agent cares mostly about the mean of $u_\gamma$ and is forgiving of loss-state outcomes; at high $\gamma$ the agent weights losses heavily, so a model whose recommended $f^\star$ produces too much loss-state pain on the validation set is penalized.

**The pedagogical punchline.** "Statistical accuracy" is not the right model-selection criterion if you are going to make a decision. The right criterion is *how well does the model's implied decision perform on data the model has not seen.* Two different agents who care about different decision losses can rationally disagree about which model is best, even when they have the same data and run the same procedure.

**Connection to notebook 2.** This is exactly the principle behind `06_Portfolio_optimization_and_copulas/2_simple_portfolio_optimization.ipynb`, where the CRPS-best statistical model (Johnson $S_U$) is not always the model whose $f^\star$ wins the validation race. There the candidates were continuous return distributions; here they are binary gambles. The lesson is the same.